# 21.20 / MP-08. Реконструкция 4D-КТ: проверка на Нике

**Статус 01.10.2026: реальная ретроспективная проверка выполнена; по MAE в области сердца на двух удержанных фазах выигрыша относительно простой интерполяции нет.** Оригиналы фаз 01 и 07 использованы только как контроль после реконструкции из уменьшенных копий. Отдельной независимой highres-съёмки и независимой анатомической разметки нет. Синтетический API smoke и preflight ниже сохранены как история.

Первичный метод: [MONAI Physio Tutorial 3](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-3-reconstruct-high-resolution-4d-ct) и [WorkflowReconstructHighres4DCT](https://project-monai.github.io/monai-physio/api/workflows.html#high-resolution-4d-ct-reconstruction). Название high-resolution само по себе не подтверждает добавление измеренной пространственной информации.


## Исходный preflight Ника (28.09.2026)

[Read-only проверка заголовков](mp08_09_nix_preflight_results.json) установила общую сетку `512×512×256`, шаг по affine `0,765625×0,765625×0,625` мм и согласованные sform/qform. В NIfTI единицы `unknown`; миллиметры следуют из DICOM-манифеста. Отдельной независимой highres-опоры нет. На момент preflight пациентская реконструкция ещё не выполнялась; последующий реальный опыт приведён ниже. Старый preflight Георга сохранён как исторический материал.


## Реальный опыт: удержанное исходное разрешение фаз 01 и 07 (01.10.2026)

**Измеренный вычислительный результат.** Фаза 00 — реальная опора на исходной сетке. Фазы 01 и 07 по отдельности сглажены (σ=0,8 вокселя) и уменьшены в 2 раза по каждой оси. В MONAI Physio MP-08 передана только уменьшенная целевая фаза; её оригинал прочитан после расчёта для оценки. Crop и область оценки определены только автоматической маской `whole_heart` фазы 00. Алгоритм: Greedy CC, `[12,6,3]`, сглаживание `3.0vox 1.0vox`, `composite_mode=reference`. Сравнение на одинаковой исходной сетке и в 1 703 924 вокселях маски фазы 00.

| Фаза | MAE обычной линейной интерполяции, HU | MAE MP-08, HU | Вывод по MAE |
|---|---:|---:|---|
| 01 | 13,07 | 25,26 | MP-08 хуже в 1,93 раза |
| 07 | 12,14 | 21,74 | MP-08 хуже в 1,79 раза |

Для фазы 07 RMSE интерполяции 29,65 HU, MP-08 29,32 HU; это отдельная метрика с иным выводом. Изображение MP-08 выглядит резче, но детали не совпадают с удержанной КТ настолько хорошо, чтобы улучшить MAE. [Полный отчёт](mp08_nix_real_holdout_report.md), [код](mp08_nix_real_holdout.py), [пофазные агрегаты 01](mp08_nix_real_holdout_phase_01.json) и [07](mp08_nix_real_holdout_phase_07.json). Локальные изображения и реконструированные объёмы находятся в исключённом из Git `runs/mp08_nix_real_holdout/`.

**Ограничение:** это реальная КТ с искусственным понижением разрешения для слепой проверки оригинала целевой фазы, а не независимое второе высокоразрешающее сканирование. Анатомическая правильность и польза для УО/ФВ не установлены. Отрицательный вывод относится к выбранным фазам, режиму `reference` и MAE в данной области; другие режимы здесь не оценены. Настройки Greedy взяты из предыдущего MP-03, где фазу 07 уже просматривали: исходные пиксели цели не входили в этот расчёт, но выбор параметров не был полностью слепым.


## Прочитанные входы и результат

**Факт о файлах, 28.09.2026.** Из внешнего производного хранилища на локальной копии D: прочитаны конверсионный манифест и только заголовки 15 NIfTI фаз испытуемого с псевдонимом Georg. Пиксели, исходные DICOM и другие зеркала не читались и не копировались. Фазы имеют сетку 512 × 512 × 256 и одинаковый affine; шаги из affine — 0.458984 × 0.458984 × 0.625 мм по конверсионному манифесту. Статус манифеста: `converted_pending_image_qc`. Сведения о физической единице в NIfTI-заголовке имеют значение `unknown`; миллиметры пока опираются на происхождение конверсии, а не на единицу NIfTI. Проверка sform/qform и совпадения манифеста с заголовками прошла. Программа и обезличенный агрегированный вывод: `mp08_09_preflight.py`, `mp08_09_preflight_results.json`.

**Временное ограничение.** Тег фаз — `NominalPercentageOfCardiacPhase`, значения 23–164; это не миллисекунды и не автоматически один равномерный механический цикл. Для Georg в проекте отмечены два различных R–R интервала 491 и 1159 мс. Привязка и выбор анализируемого интервала требуют отдельного принятия.

**Нерешённый вопрос.** Отдельный индивидуальный том с более мелкой сеткой для этой постановки не предоставлен. Использование одной из этих фаз как опоры оставляет разрешение сетки тем же. MP-03 ещё не передал принятое поле и QC складок, обратимости, границ и анатомического совмещения. Поэтому пациентская реконструкция не запускалась.

## Синтетический API smoke (28.09.2026)

**Модельный результат.** В установленном MONAI Physio 2026.09.1 `mp08_09_synthetic_smoke.py` создал два аналитических 3D фантома 32³ с шагом 1 мм и отдельную опору 48³ с шагом 0.75 мм. Известный сдвиг второго фантома — (2, −1, 0) мм. Вызван официальный `WorkflowReconstructHighres4DCT` с `RegisterImagesGreedy` в rigid-режиме и коротким расписанием [8, 4, 2]. API вернул два реконструированных объёма 43³ с шагом 0.75 мм и трансформы. Размер 43³ отражает выбранный workflow/FOV, а не совпадает автоматически с полной сеткой опоры 48³.

**Ограничение.** В нескольких запусках короткого smoke норма ошибки восстановления заданного сдвига в центре менялась примерно от 1.05 до 1.20 мм (первый запуск — 1.2004 мм); конкретное значение и вектор последнего запуска записаны в `mp08_09_synthetic_results.json`. При столь короткой оптимизации точного восстановления заданного движения **не показано**. Число не переносится на пациента и не является оценкой точности MONAI Physio в клинике. Фантом создан в памяти, медицинские данные не читались.

Для отдельной проверки точности алгоритма потребуются заранее заданные допуски, несколько сдвигов/деформаций и настройки оптимизации; этот запуск подтверждает только работоспособность данного маршрута API и фиксирует его ошибку.

## Дальнейшая проверка

Для оценки именно переноса высокочастотной информации нужен отдельный highres-том Ника того же состояния или иной независимый референс. Затем заранее задать метрики для границ, HU и анатомических структур и повторить сравнение с интерполяцией. Текущий отрицательный результат по MAE не является оценкой качества УО/ФВ.
